This page is dedicated to experiment the TFP implementation of $̂\hat{R}_{\nu}$ on Multi-Variate Normal distributions with different covariance.

In [ ]:
!pip install -Uq tfp-nightly[jax]
!pip install inference_gym
!pip install tf-nightly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 140.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.7/630.7 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 132.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 35.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6

In [ ]:
import numpy as np
from copy import deepcopy
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D

# from matplotlib.pyplot import *

import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax
from jax import random
from jax import numpy as jnp

# from inference_gym import using_jax as gym

from tensorflow_probability.substrates import jax as tfp
tfd = tfp.distributions
from inference_gym import using_jax as gym
tfb = tfp.bijectors

from tensorflow_probability.substrates import numpy as tfp_np
tfd_np = tfp_np.distributions

import matplotlib.pyplot as plt

import pandas as pd

import gc

# check if this is run on a gpu
print(jax.devices())
print(jax.default_backend())

import warnings
warnings.filterwarnings('ignore')

import psutil

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

[CudaDevice(id=0)]
gpu


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
utility_link = '/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/TFP_files/utility.py'
with open(utility_link) as f: exec(f.read())

In [ ]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [ ]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
num_chains_short = 2048
num_super_chains = 16
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

**MVN-Isotropic Covariance**

In [ ]:
num_dim = 100
# mean array:
mu = jnp.full(num_dim, 1.3)
# covariance matrix:
cov = jnp.eye(num_dim)
target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
    return random.normal(key, shape + (num_dim,))

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [ ]:
#simulation part:
MSE_constrained_list = []
MSE_naive_list = []
R_Hat_constrained_list = []
R_Hat_naive_list = []
state_list_constrained = []
state_list_naive = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)

for length in warmup_length:
  mem(f"Simulation start")
  simulation(
            keys,initialize,
            length,1,
            False, repitition,True,
            MSE_constrained_list,R_Hat_constrained_list,
            num_dim, state_list_constrained,
            mean_benchmark,var_benchmark,
            num_super_chains,num_chains_short,
            target_log_prob_fn, init_step_size)
  simulation(
            keys,initialize,
            length,1,
            True, repitition,True,
            MSE_naive_list,R_Hat_naive_list,
            num_dim, state_list_naive,
            mean_benchmark,var_benchmark,
            num_super_chains,num_chains_short,
            target_log_prob_fn, init_step_size)

Simulation start: 1581.4 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.01761559396982193
Naive initialization. Warmup Length: 10; mean of MSE is: 0.01737087033689022
Simulation start: 2226.6 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0011870389571413398
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0010458962060511112
Simulation start: 2591.0 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0005192087264731526
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0005102156428620219
Simulation start: 3115.4 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0004965230473317206
Naive initialization. Warmup Length: 40; mean of MSE is: 0.00048616837011650205
Simulation start: 3799.9 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0005208253278397024
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005108087789267302
Simulation start: 4637.1 MB
Constrained ini

In [ ]:
MSE_c_df = pd.DataFrame(MSE_constrained_list)
R_Hat_c_df = pd.DataFrame(R_Hat_constrained_list)
state_c_df = pd.DataFrame(state_list_constrained)
MSE_n_df = pd.DataFrame(MSE_naive_list)
R_Hat_n_df = pd.DataFrame(R_Hat_naive_list)
state_n_df = pd.DataFrame(state_list_naive)

In [ ]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/TFP_Iso_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/TFP_Iso_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/TFP_Iso_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/TFP_Iso_Rhat_n.pkl"
)

**MVN-AR(1) Covariance**

In [ ]:
num_dim = 100
rho = 0.5
# mean array:
mu = jnp.full(num_dim, 1.3)
# covariance matrix:
cov = [ [rho**(abs(i-j)) for j in range(num_dim)] for i in range(num_dim)]
target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=jnp.array(cov))
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
    return random.normal(key, shape + (num_dim,))
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [ ]:
#simulation part:
MSE_constrained_list = []
MSE_naive_list = []
R_Hat_constrained_list = []
R_Hat_naive_list = []
state_list_constrained = []
state_list_naive = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)

for length in warmup_length:
  mem(f"Simulation start")
  simulation(
            keys,initialize,
            length,1,
            False, repitition,True,
            MSE_constrained_list,R_Hat_constrained_list,
            num_dim, state_list_constrained,
            mean_benchmark,var_benchmark,
            num_super_chains,num_chains_short,
            target_log_prob_fn, init_step_size)
  simulation(
            keys,initialize,
            length,1,
            True, repitition,True,
            MSE_naive_list,R_Hat_naive_list,
            num_dim, state_list_naive,
            mean_benchmark,var_benchmark,
            num_super_chains,num_chains_short,
            target_log_prob_fn, init_step_size)

Simulation start: 1576.1 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.8077495694160461
Naive initialization. Warmup Length: 10; mean of MSE is: 0.8018344044685364
Simulation start: 2229.6 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.38531380891799927
Naive initialization. Warmup Length: 20; mean of MSE is: 0.4002240300178528
Simulation start: 2593.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.1312117874622345
Naive initialization. Warmup Length: 30; mean of MSE is: 0.13879069685935974
Simulation start: 3118.5 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.038554269820451736
Naive initialization. Warmup Length: 40; mean of MSE is: 0.04923897609114647
Simulation start: 3802.8 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.005273228045552969
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00796520709991455
Simulation start: 4640.1 MB
Constrained initialization. Warmup

In [ ]:
MSE_c_df = pd.DataFrame(MSE_constrained_list)
R_Hat_c_df = pd.DataFrame(R_Hat_constrained_list)
state_c_df = pd.DataFrame(state_list_constrained)
MSE_n_df = pd.DataFrame(MSE_naive_list)
R_Hat_n_df = pd.DataFrame(R_Hat_naive_list)
state_n_df = pd.DataFrame(state_list_naive)

In [ ]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/TFP_AR_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/TFP_AR_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/TFP_AR_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/TFP_AR_Rhat_n.pkl"
)